# LMF+KL Learning-Rate Ablation

This notebook mirrors the structure of `Ablations.ipynb`, but targets the extension method `lmf_KL`.

Protocol from Section 4.1 and Appendix E:

- unlearn each CoT step independently;
- use sentence-level steps, content-word/POS filtering, and FF2-only updates;
- run 5 unlearning epochs;
- use 30 pilot instances for learning-rate selection;
- hold out 20 instances for specificity;
- select the learning rate with maximum efficacy subject to `round(specificity) >= 95`.

`faithfulness` is reported for information only, matching Appendix E; it is not used to select the learning rate.

In [13]:
import json
import math
import os
import sys
from pathlib import Path

CWD = Path.cwd()
if (CWD / "unlearn.py").exists():
    ROOT = CWD
elif (CWD.parent / "unlearn.py").exists():
    ROOT = CWD.parent
else:
    ROOT = CWD
sys.path.insert(0, str(ROOT))

METHOD = "lmf_KL"
RESULTS_DIR = str(Path("lmf") / "ablation")
N_UNLEARN = 30
N_VERIFY = 20
EPOCHS = 5
RT_LAMBDA = 1.0
SPECIFICITY_THRESHOLD = 95

REFINED_LMF_LR_GRID = {
    ("LLaMA-3-3B", "openbook"): [1.5e-5, 2e-5, 2.5e-5],
    ("LLaMA-3-3B", "sqa"):      [1.5e-5, 2e-5, 2.5e-5],
    ("Phi-3", "openbook"):      [3.5e-5, 4e-5, 4.5e-5],
    ("Phi-3", "sqa"):           [6e-5, 7e-5, 8e-5, 9e-5],
}

BEST_LRS = {
    ("LLaMA-3-3B", "openbook"): [1.5e-5],
    ("LLaMA-3-3B", "sqa"):      [2.5e-5],
    ("Phi-3", "openbook"):      [4.5e-5],
    ("Phi-3", "sqa"):           [6e-5],
}

GRID = BEST_LRS

## Planned Runs

In [14]:
def iter_runs(grid, short_model=None, dataset=None, lr_values=None):
    for (model_name, dataset_name), configured_lrs in grid.items():
        if short_model is not None and model_name != short_model:
            continue
        if dataset is not None and dataset_name != dataset:
            continue
        lrs = lr_values if lr_values is not None else configured_lrs
        for lr in lrs:
            yield model_name, dataset_name, lr


planned_runs = list(iter_runs(GRID))
for short_model, dataset, lr in planned_runs:
    print(f"{short_model}\t{dataset}\tlr={lr}")
print(f"Total planned runs: {len(planned_runs)}")

LLaMA-3-3B	openbook	lr=1.5e-05
LLaMA-3-3B	sqa	lr=2.5e-05
Phi-3	openbook	lr=4.5e-05
Phi-3	sqa	lr=6e-05
Total planned runs: 4


## Run LMF+KL Pilot Points

Set one of the run toggles below to `True` when you want to launch training. Keeping the default `False` makes the notebook safe to run top-to-bottom for analysis only.

In [15]:
def run_lmf_pilot_point(
    short_model,
    dataset,
    lr,
    results_dir=RESULTS_DIR,
    n_unlearn=N_UNLEARN,
    n_verify=N_VERIFY,
    epochs=EPOCHS,
    rt_lambda=RT_LAMBDA,
):
    from repro import config as cfg
    from repro.run_repro import build_args, patched_main

    old_method = cfg.METHOD
    cfg.METHOD = METHOD
    log_suffix = "" if rt_lambda == 1.0 else f"_lambda={rt_lambda:g}"
    try:
        args = build_args(
            short_model=short_model,
            dataset=dataset,
            lr=lr,
            smoke=False,
            rt_lambda=rt_lambda,
            results_dir=results_dir,
            n_unlearn=n_unlearn,
            n_verify=n_verify,
            epochs=epochs,
            log_suffix=log_suffix,
        )
        args.method = METHOD
        patched_main(args)
    finally:
        cfg.METHOD = old_method


# Single pilot point. Edit this tuple, set RUN_SINGLE=True, then run this cell.
RUN_SINGLE = False
SINGLE_RUN = ("Phi-3", "openbook", 3e-5)

if RUN_SINGLE:
    run_lmf_pilot_point(*SINGLE_RUN)

In [16]:
# Full grid. Set RUN_GRID=True only when you are ready for a long GPU run.
RUN_GRID = False

if RUN_GRID:
    for short_model, dataset, lr in iter_runs(GRID):
        print("=" * 72)
        print(f"LMF+KL pilot: {short_model} x {dataset} lr={lr}")
        print("=" * 72)
        run_lmf_pilot_point(short_model, dataset, lr)